In [ ]:
import sys, os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/joshuaswords/netflix-data-visualization/src/small_bench/checkpoints/pre_cell_9.pickle

In [ ]:
%%cudf.pandas.profile
### cell 9 ###

# Quick feature engineering

df['count'] = 1

# Extract the first country using GPU-accelerated split with expand to avoid .str.get CPU calls
# We split once (n=1) on the first comma and take the 0th column
df['first_country'] = df['country'].str.split(',', n=1, expand=True)[0]

# Shorten common country names in one GPU replace
df['first_country'] = df['first_country'].replace({
    'United States': 'USA',
    'United Kingdom': 'UK',
    'South Korea': 'S. Korea'
})

# Map ratings to age groups with GPU replace
ratings_ages = {
    'TV-PG': 'Older Kids',  'TV-MA': 'Adults',    'TV-Y7-FV': 'Older Kids',
    'TV-Y7': 'Older Kids','TV-14': 'Teens',      'R': 'Adults',
    'TV-Y': 'Kids',        'NR': 'Adults',        'PG-13': 'Teens',
    'TV-G': 'Kids',        'PG': 'Older Kids',    'G': 'Kids',
    'UR': 'Adults',        'NC-17': 'Adults'
}

df['target_ages'] = df['rating'].replace(ratings_ages)

# Clean up the genre strings in one GPU regex replace and split into lists
# This collapses any spaces around commas into a single comma
df['genre'] = (
    df['listed_in']
      .str.replace(r"\s*,\s*", ",", regex=True)
      .str.split(',')
)